In [1]:
!pip install -U transformers faiss-gpu-cu11 ultralytics accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.0 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: a

# Pre-Processing

## Create the Dataset Class

In [2]:
import os
from PIL import Image
from torch.utils.data import Dataset

class DeepFashionDataset(Dataset):
    def __init__(self, partition_file, img_base_dir, split, transform=None, bbox_dict=None):
        """
        Args:
            partition_file (str): Path to list_eval_partition.txt
            img_base_dir (str): Path to the folder containing the 'img' directory
            split (str): One of 'train', 'gallery', or 'query'
            transform (callable, optional): PyTorch transforms to apply to the image
            bbox_dict (dict, optional): Dictionary containing bounding boxes
        """
        self.img_base_dir = img_base_dir
        self.transform = transform
        self.bbox_dict = bbox_dict
        
        # Load the partition file
        df = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
        df.columns = df.columns.str.strip()
        
        # Filter for the requested split
        self.data = df[df['evaluation_status'] == split].reset_index(drop=True)
        print(f"Initialized {split} dataset with {len(self.data)} images.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        rel_path = row['image_name']
        item_id = row['item_id']
        
        # Construct full image path and load it
        img_path = os.path.join(self.img_base_dir, rel_path)
        
        # Convert to RGB to ensure consistency (some images might be grayscale/RGBA)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # Get bounding box if available
        bbox = []
        if self.bbox_dict and rel_path in self.bbox_dict:
            bbox = self.bbox_dict[rel_path]
            
        return {
            'image': image,
            'item_id': item_id,
            'rel_path': rel_path,
            'bbox': bbox
        }

## Cropping Function for YOLO

In [3]:
import os
import random
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
from ultralytics import YOLO
from tqdm.auto import tqdm
from IPython.display import display

# --- KAGGLE DEVICE CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- PATHS ---
PARTITION_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_eval_partition.txt"
BBOX_FILE = "/kaggle/input/datasets/dveers/vr-final-ds/list_bbox_inshop.txt"
BASE_DIR = os.path.abspath("/kaggle/input/datasets/dveers/vr-final-ds")
YOLO_MODEL_PATH = '/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt'

# --- LOAD YOLO ---
print("Loading fine-tuned YOLO...")
yolo_model = YOLO(YOLO_MODEL_PATH)

def get_cropped_image(img_path, bbox=None):
    """Crops the image using bounding boxes or YOLO."""
    img = Image.open(img_path).convert('RGB')
    
    if bbox is not None and len(bbox) == 4:
        return img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
        
    # FIX: Added device targeting and half-precision for Kaggle GPU speed
    use_half = True if DEVICE == "cuda" else False
    results = yolo_model(img, conf=0.25, verbose=False, half=use_half)
    
    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.data
        best_box = boxes[boxes[:, 4].argmax()] 
        x1, y1, x2, y2 = best_box[:4].tolist()
        return img.crop((x1, y1, x2, y2))
        
    return img

# --- Quick Test ---
# test_rel_path = 'img/WOMEN/Leggings/id_00000225/03_2_side.jpg'
# test_abs_path = os.path.join(BASE_DIR, test_rel_path)
# if os.path.exists(test_abs_path):
#     cropped = get_cropped_image(test_abs_path)
#     display(cropped) 
# else:
#     print(f"Test image not found at: {test_abs_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading fine-tuned YOLO...


# Index Builders

In [4]:
# --- DUAL-GPU CONFIGURATION ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    print(f"Dual GPUs detected! Splitting the load...")
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1} # Forces BLIP onto GPU 1
else:
    print("Warning: Only 1 GPU detected. Running everything on cuda:0...")
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:0"
    BLIP_DEVICE_MAP = {"": 0}

Checking GPU availability...
Dual GPUs detected! Splitting the load...


## Part A

In [5]:
import os, json, torch, faiss, gc
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

# --- 1. DUAL-GPU CONFIGURATION ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    print(f"Dual GPUs detected! Splitting the load...")
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1} # Forces BLIP onto GPU 1
else:
    print("Warning: Only 1 GPU detected. Running everything on cuda:0...")
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"
    DEVICE_BLIP = DEVICE_YOLO
    BLIP_DEVICE_MAP = {"": 0}

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
YOLO_MODEL_PATH = "/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt"
CLIP_PATH = "openai/clip-vit-base-patch32"
INDEX_OUTPUT_PATH = "/kaggle/working/gallery_index_pretrained.faiss"
EXISTING_META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"

# --- 2. SMART-LOAD MODELS ---
print("\nLoading Models for Part A Indexing...")
if 'yolo_model' in globals():
    print("YOLO is already loaded in memory!")
else:
    yolo_model = YOLO(YOLO_MODEL_PATH)

if 'clip_model' in globals() and 'clip_processor' in globals():
    print("Pre-Trained CLIP is already loaded in memory!")
else:
    clip_processor = CLIPProcessor.from_pretrained(CLIP_PATH)
    clip_model = CLIPModel.from_pretrained(CLIP_PATH).to(DEVICE_YOLO)

def get_cropped_image(img_path):
    img = Image.open(img_path).convert('RGB')
    use_half = True if "cuda" in DEVICE_YOLO else False
    device_id = 0 if "cuda:0" in DEVICE_YOLO else "cpu"
    results = yolo_model(img, conf=0.25, verbose=False, half=use_half, device=device_id)
    if len(results[0].boxes) > 0:
        return img.crop(results[0].boxes.data[results[0].boxes.data[:, 4].argmax()][:4].tolist())
    return img

with open(EXISTING_META_PATH, "r") as f:
    metadata = json.load(f)

# --- 3. INDEXING LOOP ---
embeddings = []
print("\nBuilding Vision-Only Index for Pre-trained CLIP...")

for item in tqdm(metadata):
    abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
    if not os.path.exists(abs_path): continue
        
    cropped_img = get_cropped_image(abs_path)
    
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # Vision Only (Alpha = 1)
        clip_inputs = clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        img_embed = clip_model.visual_projection(vision_out.pooler_output)
        
        # Normalize
        img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
        embeddings.append(img_embed.to(torch.float32).cpu().numpy()[0])

index = faiss.IndexHNSWFlat(512, 32)
index.add(np.array(embeddings, dtype=np.float32))
faiss.write_index(index, INDEX_OUTPUT_PATH)
print(f"Saved Part A Index to {INDEX_OUTPUT_PATH}")

Checking GPU availability...
Dual GPUs detected! Splitting the load...

Loading Models for Part A Indexing...
YOLO is already loaded in memory!


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Building Vision-Only Index for Pre-trained CLIP...


  0%|          | 0/12612 [00:00<?, ?it/s]

Saved Part A Index to /kaggle/working/gallery_index_pretrained.faiss


## Part B

In [6]:
import os, json, torch, faiss, gc
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

ALPHAS = [0.5, 0.8]
INDEX_PATH_B_05 = "/kaggle/working/gallery_index_pretrained_alpha05.faiss"
INDEX_PATH_B_08 = "/kaggle/working/gallery_index_pretrained_alpha08.faiss"

# --- SMART-LOAD CHECK ---
if 'clip_model' in globals() and 'clip_processor' in globals():
    print("Pre-Trained CLIP is already loaded in memory!")
else:
    print("Loading Pre-Trained CLIP...")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE_YOLO)

print("\nBuilding Cross-Modal Fused Indices for Part B (Alphas: 0.5 & 0.8)...")
embeddings_05 = []
embeddings_08 = []

for item in tqdm(metadata):
    abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
    if not os.path.exists(abs_path): continue
        
    cropped_img = get_cropped_image(abs_path)
    caption = item['generated_caption']
    
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # 1. Image Embedding
        img_inputs = clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=img_inputs.pixel_values)
        img_embed = clip_model.visual_projection(vision_out.pooler_output)
        img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
        
        # 2. Text Embedding
        text_inputs = clip_processor(text=caption, return_tensors="pt", padding=True, truncation=True).to(DEVICE_YOLO)
        text_out = clip_model.text_model(**text_inputs)
        txt_embed = clip_model.text_projection(text_out.pooler_output)
        txt_embed = txt_embed / txt_embed.norm(p=2, dim=-1, keepdim=True)
        
        # 3. Double Fusion Logic 
        fused_05 = (0.5 * img_embed) + (0.5 * txt_embed)
        fused_05 = fused_05 / fused_05.norm(p=2, dim=-1, keepdim=True)
        embeddings_05.append(fused_05.to(torch.float32).cpu().numpy()[0])
        
        fused_08 = (0.8 * img_embed) + (0.2 * txt_embed)
        fused_08 = fused_08 / fused_08.norm(p=2, dim=-1, keepdim=True)
        embeddings_08.append(fused_08.to(torch.float32).cpu().numpy()[0])

# Save both HNSW Indices
index_b_05 = faiss.IndexHNSWFlat(512, 32)
index_b_05.add(np.array(embeddings_05, dtype=np.float32))
faiss.write_index(index_b_05, INDEX_PATH_B_05)

index_b_08 = faiss.IndexHNSWFlat(512, 32)
index_b_08.add(np.array(embeddings_08, dtype=np.float32))
faiss.write_index(index_b_08, INDEX_PATH_B_08)

print(f"Saved Part B Indices to {INDEX_PATH_B_05} and {INDEX_PATH_B_08}")

Pre-Trained CLIP is already loaded in memory!

Building Cross-Modal Fused Indices for Part B (Alphas: 0.5 & 0.8)...


  0%|          | 0/12612 [00:00<?, ?it/s]

Saved Part B Indices to /kaggle/working/gallery_index_pretrained_alpha05.faiss and /kaggle/working/gallery_index_pretrained_alpha08.faiss


## Part C

In [7]:
import os, json, torch, faiss, gc
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

SEEDS = [543, 45, 56]
ALPHAS = [0.5, 0.8]

print("Building Part C Indices (3 Seeds x 2 Alphas = 6 Indices total)...")

for seed in SEEDS:
    print(f"\nBuilding Indices for Seed {seed}...")
    model_path = f"/kaggle/input/models/dveers/finetuned-clip-vr-final/pytorch/default/2/finetuned_clip_full_{seed}"
    
    # Load specific seeded model to YOLO's GPU
    ft_clip_processor = CLIPProcessor.from_pretrained(model_path)
    ft_clip_model = CLIPModel.from_pretrained(model_path).to(DEVICE_YOLO)
    
    embeddings_05, embeddings_08 = [], []
    
    for item in tqdm(metadata, desc=f"Encoding Seed {seed}"):
        abs_path = os.path.join(BASE_DIR, "img", item['image_path'])
        if not os.path.exists(abs_path): continue
            
        cropped_img = get_cropped_image(abs_path)
        caption = item['generated_caption']
        
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            img_inputs = ft_clip_processor(images=cropped_img, return_tensors="pt").to(DEVICE_YOLO)
            vision_out = ft_clip_model.vision_model(pixel_values=img_inputs.pixel_values)
            img_embed = ft_clip_model.visual_projection(vision_out.pooler_output)
            img_embed = img_embed / img_embed.norm(p=2, dim=-1, keepdim=True)
            
            text_inputs = ft_clip_processor(text=caption, return_tensors="pt", padding=True, truncation=True).to(DEVICE_YOLO)
            text_out = ft_clip_model.text_model(**text_inputs)
            txt_embed = ft_clip_model.text_projection(text_out.pooler_output)
            txt_embed = txt_embed / txt_embed.norm(p=2, dim=-1, keepdim=True)
            
            fused_05 = (0.5 * img_embed) + (0.5 * txt_embed)
            fused_05 = fused_05 / fused_05.norm(p=2, dim=-1, keepdim=True)
            embeddings_05.append(fused_05.to(torch.float32).cpu().numpy()[0])
            
            fused_08 = (0.8 * img_embed) + (0.2 * txt_embed)
            fused_08 = fused_08 / fused_08.norm(p=2, dim=-1, keepdim=True)
            embeddings_08.append(fused_08.to(torch.float32).cpu().numpy()[0])

    # Save HNSW Index for Alpha 0.5
    idx_05 = faiss.IndexHNSWFlat(512, 32)
    idx_05.add(np.array(embeddings_05, dtype=np.float32))
    faiss.write_index(idx_05, f"/kaggle/working/gallery_index_finetuned_{seed}_alpha05.faiss")
    
    # Save HNSW Index for Alpha 0.8
    idx_08 = faiss.IndexHNSWFlat(512, 32)
    idx_08.add(np.array(embeddings_08, dtype=np.float32))
    faiss.write_index(idx_08, f"/kaggle/working/gallery_index_finetuned_{seed}_alpha08.faiss")
    
    print(f"Saved Seed {seed} Indices!")
    
    del ft_clip_model, ft_clip_processor, idx_05, idx_08
    gc.collect()
    if torch.cuda.device_count() >= 2:
        with torch.cuda.device(0): torch.cuda.empty_cache() 
    else:
        torch.cuda.empty_cache()

Building Part C Indices (3 Seeds x 2 Alphas = 6 Indices total)...

Building Indices for Seed 543...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 543:   0%|          | 0/12612 [00:00<?, ?it/s]

Saved Seed 543 Indices!

Building Indices for Seed 45...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 45:   0%|          | 0/12612 [00:00<?, ?it/s]

Saved Seed 45 Indices!

Building Indices for Seed 56...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding Seed 56:   0%|          | 0/12612 [00:00<?, ?it/s]

Saved Seed 56 Indices!


## Zip and Download Indices

In [8]:
import os
import zipfile
from IPython.display import display, FileLink

# --- Configuration ---
WORKING_DIR = "/kaggle/working"
ZIP_NAME = "all_faiss_indices.zip"
ZIP_PATH = os.path.join(WORKING_DIR, ZIP_NAME)

print("Scanning /kaggle/working/ for FAISS indices...")

# Find all files starting with 'gallery_index_' and ending with '.faiss'
faiss_files = [f for f in os.listdir(WORKING_DIR) if f.startswith("gallery_index_") and f.endswith(".faiss")]

if not faiss_files:
    print("❌ No FAISS indices found! Make sure your index builders have finished running.")
else:
    print(f"Found {len(faiss_files)} files. Compressing now...")
    
    # 1. Zip the files together using standard deflation
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in faiss_files:
            print(f" 📦 Adding: {file}")
            # arcname prevents it from zipping the entire folder tree, keeping it flat
            zipf.write(os.path.join(WORKING_DIR, file), arcname=file)
            
    # 2. Calculate the final archive size
    file_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"\n✅ Zipping complete! Final archive size: {file_size_mb:.2f} MB")
    
    # 3. Generate the stable download link
    print("\n⬇️ Click the link below to download your archive directly:")
    display(FileLink(ZIP_NAME))

Scanning /kaggle/working/ for FAISS indices...
Found 9 files. Compressing now...
 📦 Adding: gallery_index_finetuned_45_alpha05.faiss
 📦 Adding: gallery_index_finetuned_56_alpha05.faiss
 📦 Adding: gallery_index_pretrained.faiss
 📦 Adding: gallery_index_finetuned_543_alpha05.faiss
 📦 Adding: gallery_index_finetuned_56_alpha08.faiss
 📦 Adding: gallery_index_finetuned_543_alpha08.faiss
 📦 Adding: gallery_index_pretrained_alpha08.faiss
 📦 Adding: gallery_index_pretrained_alpha05.faiss
 📦 Adding: gallery_index_finetuned_45_alpha08.faiss

✅ Zipping complete! Final archive size: 215.71 MB

⬇️ Click the link below to download your archive directly:


/kaggle/working/all_faiss_indices.zip

# Ablation Study

In [9]:
import pandas as pd
import numpy as np
import torch
import os
import faiss
from tqdm.auto import tqdm
from IPython.display import display, HTML

# 1. Point to your partition file

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
PARTITION_FILE = os.path.join(BASE_DIR, "list_eval_partition.txt")

print("Loading query partition...")

# 2. Load the text file into a pandas DataFrame
df_part = pd.read_csv(PARTITION_FILE, sep=r'\s+', skiprows=1)

# 3. Clean the column names (scrubs hidden Windows characters)
df_part.columns = df_part.columns.str.strip()

# 4. Filter to create query_df (grabbing ONLY the 'query' status images)
query_df = df_part[df_part['evaluation_status'] == 'query'].copy()

# 5. Clean the image paths just to be safe
query_df['image_name'] = query_df['image_name'].str.strip()

print(f"Success! Loaded {len(query_df)} Query images ready for testing.")

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

Loading query partition...
Success! Loaded 14218 Query images ready for testing.


## Part A‎ 

In [10]:
import os
import json
import torch
import faiss
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel
from IPython.display import display, HTML

# --- 1. CONFIGURATION ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    DEVICE_YOLO = "cuda:0"
else:
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
INDEX_PATH_A = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained.faiss" 
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"
k_values = [5, 10, 15]

# --- 2. LOAD METADATA ---
with open(META_PATH, "r", encoding='utf-8') as f:
    metadata = json.load(f)

# --- 3. METRICS FUNCTION ---
def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

# --- 4. LOAD MODELS ---
print("Loading Pre-Trained CLIP for Part A...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE_YOLO)

index = faiss.read_index(INDEX_PATH_A)
total_metrics = {'Recall': {5: 0, 10: 0, 15: 0}, 'NDCG': {5: 0, 10: 0, 15: 0}, 'mAP': {5: 0, 10: 0, 15: 0}}
valid_queries = 0

# --- 5. EVALUATION LOOP ---
print("\n--- Running Evaluation for Part A (Vision-Only) ---")
for idx, row in tqdm(query_df.iterrows(), total=len(query_df)):
    abs_path = os.path.join(BASE_DIR, "img", row['image_name'])
    if not os.path.exists(abs_path): continue
    valid_queries += 1
    
    cropped_query = get_cropped_image(abs_path) 
    
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # Vision Only (Alpha = 1)
        clip_inputs = clip_processor(images=cropped_query, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        query_embed = clip_model.visual_projection(vision_out.pooler_output)
        query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
        
        # Search the Pretrained Vision-Only Index
        distances, indices = index.search(query_embed.to(torch.float32).cpu().numpy(), 15)
        candidates = [metadata[i] for i in indices[0]]
        
    retrieved_item_ids = [c['item_id'] for c in candidates]
    metrics = calculate_metrics(retrieved_item_ids, row['item_id'], k_values)
    
    for m in ['Recall', 'NDCG', 'mAP']:
        for k in k_values: total_metrics[m][k] += metrics[m][k]

# --- 6. OUTPUT RESULTS ---
print("\n" + "="*50 + "\nCONDITION A: Vision-Only Baseline (Alpha = 1)\n" + "="*50)
results_df = pd.DataFrame(total_metrics) / valid_queries
results_df.index = [f"@{k}" for k in k_values]
display(HTML(results_df.to_html(classes='table table-striped table-bordered text-center', float_format="%.4f")))

Checking GPU availability...
Loading Pre-Trained CLIP for Part A...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Evaluation for Part A (Vision-Only) ---


  0%|          | 0/14218 [00:00<?, ?it/s]


CONDITION A: Vision-Only Baseline (Alpha = 1)


,Recall,NDCG,mAP
@5,0.4240,0.4444,0.3193
@10,0.4881,0.5048,0.3139
@15,0.5224,0.5364,0.3069


## Part B‎ 

In [11]:
import os
import json
import torch
import faiss
import gc
import pandas as pd
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from IPython.display import display, HTML

# --- 1. CONFIGURATION & DEVICES ---
print("Checking GPU availability...")
if torch.cuda.device_count() >= 2:
    DEVICE_YOLO = "cuda:0"
    DEVICE_BLIP = "cuda:1"
    BLIP_DEVICE_MAP = {"": 1}
else:
    DEVICE_YOLO = "cuda:0" if torch.cuda.is_available() else "cpu"
    DEVICE_BLIP = DEVICE_YOLO
    BLIP_DEVICE_MAP = {"": 0}

BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"
INDEX_PATH_B_05 = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained_alpha05.faiss"
INDEX_PATH_B_08 = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices/gallery_index_pretrained_alpha08.faiss"
ALPHAS = [0.5, 0.8]
k_values = [5, 10, 15]

# --- 2. LOAD METADATA & METRICS ---
with open(META_PATH, "r", encoding='utf-8') as f:
    metadata = json.load(f)

def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

# --- 3. LOAD MODELS ---
print(f"Loading Pre-Trained CLIP to {DEVICE_YOLO}...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE_YOLO)

print(f"Loading Frozen BLIP-2 to {DEVICE_BLIP}...")
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", 
    torch_dtype=torch.float16, 
    device_map=BLIP_DEVICE_MAP
)

# --- 4. LOAD INDICES ---
index_05 = faiss.read_index(INDEX_PATH_B_05)
index_08 = faiss.read_index(INDEX_PATH_B_08)

# --- 5. EVALUATION LOOP ---
print("\nRunning ULTRA-FAST Evaluation for Part B (Processing both Alphas simultaneously)...")

total_metrics = {alpha: {'Recall': {5: 0, 10: 0, 15: 0}, 'NDCG': {5: 0, 10: 0, 15: 0}, 'mAP': {5: 0, 10: 0, 15: 0}} for alpha in ALPHAS}
valid_queries = 0

for idx, row in tqdm(query_df.iterrows(), total=len(query_df)):
    abs_path = os.path.join(BASE_DIR, "img", row['image_name'])
    if not os.path.exists(abs_path): continue
    valid_queries += 1
    
    cropped_query = get_cropped_image(abs_path)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        
        # 1. FAISS Query Embed (GPU 0)
        clip_inputs = clip_processor(images=cropped_query, return_tensors="pt").to(DEVICE_YOLO)
        vision_out = clip_model.vision_model(pixel_values=clip_inputs.pixel_values)
        query_embed = clip_model.visual_projection(vision_out.pooler_output)
        query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
        query_np = query_embed.to(torch.float32).cpu().numpy()
        
        # 2. Search BOTH indices simultaneously
        _, idx_05 = index_05.search(query_np, 15)
        _, idx_08 = index_08.search(query_np, 15)
        
        candidates_05 = [metadata[i] for i in idx_05[0]]
        candidates_08 = [metadata[i] for i in idx_08[0]]
        
        # 3. Combine unique candidates to save BLIP-2 compute time!
        unique_candidates_dict = {c['faiss_id']: c for c in candidates_05 + candidates_08}
        unique_candidates = list(unique_candidates_dict.values())
        
        # 4. BLIP-2 Batched Re-Ranking (GPU 1)
        pixel_values = blip_processor(images=cropped_query, return_tensors="pt").pixel_values.to(DEVICE_BLIP, torch.float16)
        pixel_values = pixel_values.expand(len(unique_candidates), -1, -1, -1)
        
        captions = [c['generated_caption'] for c in unique_candidates]
        text_inputs = blip_processor.tokenizer(captions, return_tensors="pt", padding=True, truncation=True).to(DEVICE_BLIP)
        
        input_ids, attention_mask = text_inputs.input_ids, text_inputs.attention_mask
        labels = input_ids.clone()
        labels[labels == blip_processor.tokenizer.pad_token_id] = -100
        
        outputs = blip_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_matrix = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
        loss_matrix = loss_matrix.view(shift_labels.size())
        mask = (shift_labels != -100)
        losses = ((loss_matrix * mask).sum(dim=1) / mask.sum(dim=1)).tolist()
        
        # Create a fast lookup dictionary for the scores
        score_lookup = {unique_candidates[i]['faiss_id']: losses[i] for i in range(len(unique_candidates))}
        
        # 5. Split, sort, and calculate metrics for Alpha 0.5
        reranked_05 = [(score_lookup[c['faiss_id']], c) for c in candidates_05]
        reranked_05.sort(key=lambda x: x[0])
        final_05 = [item[1]['item_id'] for item in reranked_05]
        metrics_05 = calculate_metrics(final_05, row['item_id'], k_values)
        for m in ['Recall', 'NDCG', 'mAP']:
            for k in k_values: total_metrics[0.5][m][k] += metrics_05[m][k]
                
        # 6. Split, sort, and calculate metrics for Alpha 0.8
        reranked_08 = [(score_lookup[c['faiss_id']], c) for c in candidates_08]
        reranked_08.sort(key=lambda x: x[0])
        final_08 = [item[1]['item_id'] for item in reranked_08]
        metrics_08 = calculate_metrics(final_08, row['item_id'], k_values)
        for m in ['Recall', 'NDCG', 'mAP']:
            for k in k_values: total_metrics[0.8][m][k] += metrics_08[m][k]

# --- 6. OUTPUT FINAL RESULTS FOR BOTH ALPHAS ---
for alpha in ALPHAS:
    print(f"\nCONDITION B: Frozen CLIP + BLIP-2 (Alpha = {alpha})")
    results_df = pd.DataFrame(total_metrics[alpha]) / valid_queries
    results_df.index = [f"@{k}" for k in k_values]
    display(HTML(results_df.to_html(classes='table table-striped table-bordered text-center', float_format="%.4f")))

Checking GPU availability...
Loading Pre-Trained CLIP to cuda:0...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Frozen BLIP-2 to cuda:0...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

...

  0%|          | 0/14218 [00:00<?, ?it/s]


CONDITION B: Frozen CLIP + BLIP-2 (Alpha = 0.5)


,Recall,NDCG,mAP
@5,0.2006,0.1393,0.0934
@10,0.3592,0.2249,0.1109
@15,0.4769,0.2941,0.1168



CONDITION B: Frozen CLIP + BLIP-2 (Alpha = 0.8)


,Recall,NDCG,mAP
@5,0.2281,0.1632,0.1080
@10,0.4050,0.2641,0.1267
@15,0.5308,0.3469,0.1327


## Part C - Done in VR_Final_C.ipynb